# Initial Data Inspection

This notebook explores the ListenBrainz listening history data and assesses whether it is suitable for building an album-level recommender system.

The main questions are:

1. What information is available in the raw listening records?
2. Can listens be reliably mapped to albums?
3. Is there enough user–album interaction data to support collaborative filtering?

In [18]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)

## 1. Inspect the raw data

A small number of records are loaded first to understand the nested JSON structure and identify the fields required for an album recommender.

In [19]:
listens_path = Path("..") / ".." / "data" / "2026" / "7.listens"
listens = []

# Open the listens file in read mode with UTF-8 encoding and read the first 5 records
with listens_path.open("r", encoding="utf-8") as listens_file:
    for _ in range(5):
        line = listens_file.readline()

        if not line:
            break

        listens.append(json.loads(line))

In [20]:
listens_df = pd.json_normalize(listens)

listens_df[
    [
        "user_id",
        "timestamp",
        "track_metadata.track_name",
        "track_metadata.artist_name",
        "track_metadata.release_name",
        "recording_msid",
    ]
]

,user_id,timestamp,track_metadata.track_name,track_metadata.artist_name,track_metadata.release_name,recording_msid
0,137288,1785110317,SWIM,BTS,SWIM (Underwater Remix),a37d9221-0717-4d02-8dd4-91f9f218f3d0
1,108691,1785110287,Sneaky Snitch,Kevin Macleod,Mystery,9a734ac1-2b04-4e9e-bf38-9ea0f8057229
2,70517,1785110251,Impostor Syndrome,Sidney Gish,NaN,83c63b5e-aac5-4748-ad62-930409c3e58b
3,137098,1785110327,SWIM with V (Electronic Remix),BTS,KEEP SWIMMING,c7c3e688-5579-44de-8420-9f7827492098
4,1377,1785110274,To Live Is to Fly,Townes Van Zandt,Rear View Mirror,8dcb1b83-b61a-4a21-b41a-b051e9503d28


In [21]:
column_names = {
    "track_metadata.track_name": "track_name",
    "track_metadata.artist_name": "artist_name",
    "track_metadata.release_name": "release_name",
}

listens_clean = (
    listens_df[
        [
            "user_id",
            "timestamp",
            "track_metadata.track_name",
            "track_metadata.artist_name",
            "track_metadata.release_name",
            "recording_msid",
        ]
    ]
    .rename(columns=column_names)
    .copy()
)

listens_clean

,user_id,timestamp,track_name,artist_name,release_name,recording_msid
0,137288,1785110317,SWIM,BTS,SWIM (Underwater Remix),a37d9221-0717-4d02-8dd4-91f9f218f3d0
1,108691,1785110287,Sneaky Snitch,Kevin Macleod,Mystery,9a734ac1-2b04-4e9e-bf38-9ea0f8057229
2,70517,1785110251,Impostor Syndrome,Sidney Gish,NaN,83c63b5e-aac5-4748-ad62-930409c3e58b
3,137098,1785110327,SWIM with V (Electronic Remix),BTS,KEEP SWIMMING,c7c3e688-5579-44de-8420-9f7827492098
4,1377,1785110274,To Live Is to Fly,Townes Van Zandt,Rear View Mirror,8dcb1b83-b61a-4a21-b41a-b051e9503d28


## 2. Assess album metadata availability

Before constructing album-level interactions, I first check whether release information is available for most listening records.

In [22]:
sample_size = 1000000
listens = []

with listens_path.open("r", encoding="utf-8") as listens_file:
    for _ in range(sample_size):
        line = listens_file.readline()

        if not line:
            break

        listens.append(json.loads(line))

large_sample_df = pd.json_normalize(listens)

large_sample_df.shape

(1000000, 81)

In [23]:
useful_columns = [
    "user_id",
    "timestamp",
    "track_metadata.track_name",
    "track_metadata.artist_name",
    "track_metadata.release_name",
    "recording_msid",
]

large_sample_clean = (
    large_sample_df[useful_columns]
    .rename(columns=column_names)
    .copy()
)

large_sample_clean.head()

,user_id,timestamp,track_name,artist_name,release_name,recording_msid
0,137288,1785110317,SWIM,BTS,SWIM (Underwater Remix),a37d9221-0717-4d02-8dd4-91f9f218f3d0
1,108691,1785110287,Sneaky Snitch,Kevin Macleod,Mystery,9a734ac1-2b04-4e9e-bf38-9ea0f8057229
2,70517,1785110251,Impostor Syndrome,Sidney Gish,NaN,83c63b5e-aac5-4748-ad62-930409c3e58b
3,137098,1785110327,SWIM with V (Electronic Remix),BTS,KEEP SWIMMING,c7c3e688-5579-44de-8420-9f7827492098
4,1377,1785110274,To Live Is to Fly,Townes Van Zandt,Rear View Mirror,8dcb1b83-b61a-4a21-b41a-b051e9503d28


In [24]:
release_coverage = large_sample_clean["release_name"].notna().mean()

print(f"Listens with a release name: {release_coverage:.1%}")

Listens with a release name: 97.8%


Release names are available for 97.8% of listens in the initial sample, suggesting that missing album data should not prevent an album-level analysis.

## 3. Define an initial album identifier

`release_name` alone cannot uniquely identify an album because different artists may release records with the same title.

For this initial analysis, an album is therefore represented by the combination of artist name and release name.

This is a provisional identifier: different editions of the same album may still appear separately, so MusicBrainz release identifiers may be investigated later.

In [26]:
large_sample_clean["album_key"] = (
    large_sample_clean["artist_name"].str.strip().str.lower()
    + " — "
    + large_sample_clean["release_name"].str.strip().str.lower()
)

In [27]:
large_sample_clean[
    ["artist_name", "release_name", "album_key"]
].head()

,artist_name,release_name,album_key
0,BTS,SWIM (Underwater Remix),bts — swim (underwater remix)
1,Kevin Macleod,Mystery,kevin macleod — mystery
2,Sidney Gish,NaN,NaN
3,BTS,KEEP SWIMMING,bts — keep swimming
4,Townes Van Zandt,Rear View Mirror,townes van zandt — rear view mirror


## 4. Investigate sampling bias

Initial user–album statistics from the first 1,000,000 rows suggested that collaborative filtering might be unsuitable because the interaction data appeared extremely sparse.

However, the distribution of listens revealed that a small number of users contributed a very large proportion of the sample. This suggests that the source file is not randomly ordered, meaning that the first n rows are not representative of the full dataset.

I therefore avoid drawing modelling conclusions from this contiguous sample and instead aggregate interactions across the complete file.

In [38]:
listens_per_user = large_sample_clean.groupby("user_id").size()

listens_per_user.sort_values(ascending=False).head(10)

user_id
148138    396100
147897    186077
9104      167172
21018      53559
148294     11283
110873      6677
138435       967
139287       844
137627       815
139827       668
dtype: int64

The initial sample is highly concentrated: a small number of users account for a very large proportion of the first 1,000,000 records. This indicates that a contiguous block from the start of the file is unlikely to be representative of the wider dataset.

## 5. Construct user–album interactions from the full dataset

Because the raw file contains almost five million JSON records, it is processed sequentially rather than fully normalised into memory.

Each valid listen increments the count for a `(user, album)` pair. The resulting table represents implicit feedback: repeated listening is treated as evidence of user interest rather than an explicit rating.

In [ ]:
# Count how many listens each user has for each album
user_album_counts = Counter()

valid_listens = 0

with listens_path.open("r", encoding="utf-8") as listens_file:

    for line_number, line in enumerate(listens_file, start=1):

        record = json.loads(line)

        user_id = record.get("user_id")

        track_metadata = record.get("track_metadata", {})
        artist_name = track_metadata.get("artist_name")
        release_name = track_metadata.get("release_name")

        # Ignore listens where we cannot identify an album
        if user_id is None or not artist_name or not release_name:
            continue

        # Temporary album identifier
        album_identifier = (
            artist_name.strip().lower(),
            release_name.strip().lower()
        )

        user_album_counts[(user_id, album_identifier)] += 1
        valid_listens += 1

        # Optional progress update
        if line_number % 500_000 == 0:
            print(f"Processed {line_number:,} listens")

Processed 500,000 listens
Processed 1,000,000 listens
Processed 1,500,000 listens
Processed 2,000,000 listens
Processed 2,500,000 listens
Processed 3,000,000 listens
Processed 3,500,000 listens
Processed 4,000,000 listens
Processed 4,500,000 listens


In [ ]:
user_album_interactions = pd.DataFrame(
    [
        {
            "user_id": user_id,
            "artist_name": album_identifier[0],
            "release_name": album_identifier[1],
            "album_key": f"{album_identifier[0]} — {album_identifier[1]}",
            "listen_count": listen_count,
        }
        for (user_id, album_identifier), listen_count in user_album_counts.items()
    ]
)

user_album_interactions.head()

,user_id,artist_name,release_name,album_key,listen_count
0,137288,bts,swim (underwater remix),bts — swim (underwater remix),35
1,108691,kevin macleod,mystery,kevin macleod — mystery,2
2,137098,bts,keep swimming,bts — keep swimming,138
3,1377,townes van zandt,rear view mirror,townes van zandt — rear view mirror,1
4,48676,blackpink,jump,blackpink — jump,1


## 6. Assess collaborative filtering viability

### 6.1 Overall interaction size

In [31]:
n_users = user_album_interactions["user_id"].nunique()

n_albums = (
    user_album_interactions[
        ["artist_name", "release_name"]
    ]
    .drop_duplicates()
    .shape[0]
)

n_interactions = len(user_album_interactions)

print(f"Valid listens: {valid_listens:,}")
print(f"Unique users: {n_users:,}")
print(f"Unique albums: {n_albums:,}")
print(f"Unique user-album interactions: {n_interactions:,}")

Valid listens: 4,743,303
Unique users: 19,346
Unique albums: 453,146
Unique user-album interactions: 716,358


The full dataset produces a substantially larger user–album interaction graph than suggested by the initial contiguous sample, motivating a more detailed assessment of interaction depth and cross-user overlap.

### 6.2 User listening depth

In [32]:
albums_per_user = (
    user_album_interactions
    .groupby("user_id")
    .size()
)

albums_per_user.describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    19346.000000
mean        37.028740
std        663.951465
min          1.000000
50%         11.000000
90%         55.000000
95%         81.000000
99%        165.550000
max      72727.000000
dtype: float64

In [33]:
user_thresholds = [5, 10, 20]

for threshold in user_thresholds:

    users_remaining = (
        albums_per_user >= threshold
    ).sum()

    percentage = (
        users_remaining
        / len(albums_per_user)
        * 100
    )

    print(
        f"Users with >= {threshold} albums: "
        f"{users_remaining:,} "
        f"({percentage:.1f}%)"
    )

Users with >= 5 albums: 13,715 (70.9%)
Users with >= 10 albums: 10,403 (53.8%)
Users with >= 20 albums: 6,537 (33.8%)


The median user has interacted with 11 distinct albums. Around 71% of users have listened to at least five albums and 54% have listened to at least ten, indicating that a substantial proportion of users have enough history for personalised recommendation.

### 6.3 Album overlap between users

In [ ]:
users_per_album = (
    user_album_interactions
    .groupby("album_key")["user_id"]
    .nunique()
)

users_per_album.describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    453146.000000
mean          1.580855
std           4.191362
min           1.000000
50%           1.000000
90%           2.000000
95%           3.000000
99%          11.000000
max         750.000000
Name: user_id, dtype: float64

In [35]:
album_thresholds = [2, 5, 10]

for threshold in album_thresholds:

    albums_remaining = (
        users_per_album >= threshold
    ).sum()

    percentage = (
        albums_remaining
        / len(users_per_album)
        * 100
    )

    print(
        f"Albums with >= {threshold} users: "
        f"{albums_remaining:,} "
        f"({percentage:.1f}%)"
    )

Albums with >= 2 users: 77,496 (17.1%)
Albums with >= 5 users: 15,472 (3.4%)
Albums with >= 10 users: 5,264 (1.2%)


Most releases occur for only one user. However, 77,496 albums are shared by at least two users and more than 15,000 are shared by at least five, providing a sizeable subset with cross-user interaction information.

### 6.4 Preliminary filtering

Finds the number of users or albums tht have enough interaction history to be useful for collaborative filtering.

In [36]:
active_users = albums_per_user[
    albums_per_user >= 5
].index

shared_albums = users_per_album[
    users_per_album >= 2
].index

filtered_interactions = user_album_interactions[
    user_album_interactions["user_id"].isin(active_users)
    & user_album_interactions["album_key"].isin(shared_albums)
].copy()

print(
    f"Users remaining: "
    f"{filtered_interactions['user_id'].nunique():,}"
)

print(
    f"Albums remaining: "
    f"{filtered_interactions['album_key'].nunique():,}"
)

print(
    f"Interactions remaining: "
    f"{len(filtered_interactions):,}"
)

Users remaining: 13,209
Albums remaining: 77,462
Interactions remaining: 333,535


## Conclusion

The initial sample substantially overstated the sparsity of the user–album interaction structure because the ListenBrainz file is not randomly ordered.

Across the full dataset, the median user interacted with 11 distinct albums, and around 71% of users interacted with at least five. Although album popularity is highly long-tailed, more than 77,000 albums are shared by at least two users.

Applying preliminary thresholds of at least five albums per user and at least two users per album leaves approximately **13,000 users, 77,000 albums and 333,000 user–album interactions**.

These results support proceeding with collaborative filtering as an initial modelling approach.